In [ ]:
##Causal inference with Double/Debiased Machine Learning (DML) for the effect of
## Midwife-Led Continuity Care (MLCC) on preterm birth, North Shoa Zone, Ethiopia.
   ##  Step 1  Load + clean data
    ## Step 2  Table 1  : chi-square comparison of preterm vs term births
    ## Step 3  Fig 3-5  : predictive model (accuracy, ROC/AUC, feature importance), PREDICTION ONLY, not used for causal adjustment
    ## Step 4  Fig S1   : propensity-score overlap (positivity check)
    ## Step 5  Table 2  : DML ATE (PLR, PLIV, IRM, IIVM x Lasso/RF/Tree/XGBoost)
    ## Step 6  Table 3  : DML quantile treatment effects (QTE)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
log = logging.getLogger("mlcc_dml")
# Baseline (pre-treatment) covariates ONLY - used in every DML nuisance model.
# Post-treatment / post-delivery variables (Apgar, birth weight, NICU, PNC,
# breast-feeding initiation, labour events, ...) are deliberately excluded.
BASELINE_COVARIATES = [
    "Income_level", "age_merrage", "Last_menstral", "Gravidity", "Parity",
    "Histry_abortion", "Number_healtyfacilty", "Birth_interval",
    "Previous_surgery", "Female_mulation", "Isomaization", "Residence",
    "Bank_account", "Weight", "Martial_status", "Tetan_dose", "Foliac_acid",
    "Nutrion", "counlig", "Blood_group_RH", "Syphilis_s", "urinep",
    "Pregnancy_inducedhypertension",
]

# Columns that define / leak the outcome or follow it -> never used as features
# in the *predictive* model (Figures 3-5).
OUTCOME_LEAKAGE = [GESTATIONAL_AGE, "pretmba", "Preterm_labour", "Y_preterm"]

# Table 1 variables: (column, label)
TABLE1_VARS = [
    ("Residence", "Residence (1=urban, 2=rural)"),
    ("Foliac_acid", "Folic acid / iron (1=given, 2=not given)"),
    ("Inttion_breast", "Breast-feeding initiation (0=after 1h, 1=within 1h)"),
    ("Allocation_mother", "Allocated group (0=other, 1=midwife-led)"),
    ("Pregnancy_inducedhypertension", "Pregnancy-induced hypertension"),
    ("Apgar_score", "5-min Apgar <= 7"),
    ("Vacuum_baby", "Vacuum delivery"),
    ("Blood_group_RH", "Blood group / Rh factor"),
    ("Presentation", "Fetal presentation"),
    ("Birth_status", "Birth status"),
    ("Admitted_intensive", "NICU admission"),
]
# ==================================
# 1. DATA
# =================================
def clean_names(columns) -> list[str]:
    """Replace every run of non-alphanumeric characters by '_' (strips ’ etc.)."""
    return [re.sub(r"\W+", "_", str(c)).strip("_") for c in columns]


def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")
    df = pd.read_excel(path) if path.suffix.lower() in {".xlsx", ".xls"} else pd.read_csv(path)
    df.columns = clean_names(df.columns)
    for col in (GESTATIONAL_AGE, TREATMENT):
        if col not in df.columns:
            raise KeyError(f"Required column '{col}' missing. Found: {list(df.columns)[:10]}...")
    # Outcome: preterm (<37 weeks)=1, term=0  (coding 1/2 in the raw data)
    df["Y_preterm"] = (df[GESTATIONAL_AGE] == 1).astype(int)
    # Treatment: MLCC=1, other professional care=0
    df["D_mlcc"] = df[TREATMENT].astype(int)
    # Basic integrity checks
    if df.isna().any().any():
        n_miss = int(df.isna().sum().sum())
        log.warning("%d missing cells found; median-imputing numeric columns.", n_miss)
        df = df.fillna(df.median(numeric_only=True))
    missing = [c for c in BASELINE_COVARIATES if c not in df.columns]
    if missing:
        raise KeyError(f"Baseline covariates not in data: {missing}")
    leak = {GESTATIONAL_AGE, "Y_preterm", "D_mlcc", TREATMENT} & set(BASELINE_COVARIATES)
    if leak:
        raise ValueError(f"Outcome/treatment appear among covariates: {leak}")
    log.info("Loaded n=%d, p=%d | preterm=%d (%.1f%%) | MLCC=%d (%.1f%%)",
             len(df), df.shape[1], df.Y_preterm.sum(), 100 * df.Y_preterm.mean(),
             df.D_mlcc.sum(), 100 * df.D_mlcc.mean())
    return df

# ===============================================
# 2. TABLE 1  - descriptive chi-square tests
# ================================================
def table1(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col, label in TABLE1_VARS:
        if col not in df.columns:
            continue
        ct = pd.crosstab(df[col], df["Y_preterm"]).reindex(columns=[1, 0], fill_value=0)
        chi2, p, _, _ = chi2_contingency(ct.values, correction=False)
        for i, (level, r) in enumerate(ct.iterrows()):
            rows.append({
                "Variable": label if i == 0 else "",
                "Category": level,
                "Preterm n (%)": f"{r[1]} ({100 * r[1] / ct[1].sum():.1f})",
                "Term n (%)": f"{r[0]} ({100 * r[0] / ct[0].sum():.1f})",
                "Chi-square": round(chi2, 3) if i == 0 else "",
                "P-value": round(p, 4) if i == 0 else "",
            })
    return pd.DataFrame(rows)
# ========================================
# 3. PREDICTIVE MODEL  (Figures 3-5) 
# ========================================
def predictive_model(df: pd.DataFrame, out: Path) -> None:
    feats = [c for c in df.columns if c not in OUTCOME_LEAKAGE + ["D_mlcc"]]
    X, y = df[feats], df["Y_preterm"]
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                random_state=SEED, n_jobs=N_JOBS).fit(Xtr, ytr)
    p_te = rf.predict_proba(Xte)[:, 1]
    acc_tr = accuracy_score(ytr, rf.predict(Xtr))
    acc_te = accuracy_score(yte, rf.predict(Xte))
    auc = roc_auc_score(yte, p_te)

    # bootstrap 95% CI for AUC
    rng = np.random.default_rng(SEED)
    boots = []
    yv = yte.values
    for _ in range(1000):
        idx = rng.integers(0, len(yv), len(yv))
        if len(np.unique(yv[idx])) == 2:
            boots.append(roc_auc_score(yv[idx], p_te[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    log.info("Predictive RF | accuracy train=%.3f test=%.3f | AUC=%.3f (95%% CI %.3f-%.3f)",
             acc_tr, acc_te, auc, lo, hi)
    pd.DataFrame({"metric": ["accuracy_train", "accuracy_test", "auc_test", "auc_ci_low", "auc_ci_high"],
                  "value": [acc_tr, acc_te, auc, lo, hi]}).to_csv(out / "fig3_4_predictive_performance.csv", index=False)
    # Figure 4 - ROC
    fpr, tpr, _ = roc_curve(yte, p_te)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, color="tab:blue", label=f"Random forest (AUC = {auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", label="No discrimination (AUC = 0.5)")
    ax.set(xlabel="1 - Specificity", ylabel="Sensitivity", title="ROC curve: preterm birth (test set)")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(out / "fig4_roc.png", dpi=300)
    plt.close(fig)

    # Figure 5 - normalised importance (predictive only!)
    imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
    imp_norm = (imp / imp.max()).head(17)
    imp_norm.to_csv(out / "fig5_feature_importance.csv", header=["normalised_importance"])
    fig, ax = plt.subplots(figsize=(6, 6))
    imp_norm[::-1].plot.barh(ax=ax, color="tab:green")
    ax.set(xlabel="Normalised importance (predictive only, not causal)",
           title="Top 17 predictors of preterm birth")
    fig.tight_layout()
    fig.savefig(out / "fig5_feature_importance.png", dpi=300)
    plt.close(fig)


# ============================
# 4. LEARNERS
# =============================
def _tuned(est, grid):
    return GridSearchCV(est, grid, cv=5, n_jobs=N_JOBS)
def make_learners(quick: bool = False) -> dict[str, dict]:
    L: dict[str, dict] = {}
    # Lasso: LassoCV (regression) / L1 logistic (classification); both pick penalty by CV
    L["Lasso"] = {
        "reg": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=SEED, max_iter=20000)),
        "clf": make_pipeline(StandardScaler(),
                             LogisticRegressionCV(penalty="l1", solver="saga", cv=5, max_iter=5000,
                                                  random_state=SEED, scoring="neg_log_loss")),
    }

    rf_grid = {"max_depth": [3, 6], "min_samples_leaf": [5, 10]} if not quick else {"max_depth": [4]}
    L["Random forest"] = {
        "reg": _tuned(RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1), rf_grid),
        "clf": _tuned(RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=1), rf_grid),
    }

    tr_grid = {"max_depth": [2, 3, 5], "min_samples_leaf": [10, 20]} if not quick else {"max_depth": [3]}
    L["Reg. tree"] = {
        "reg": _tuned(DecisionTreeRegressor(random_state=SEED), tr_grid),
        "clf": _tuned(DecisionTreeClassifier(random_state=SEED), tr_grid),
    }

    try:  # XGBoost is optional
        from xgboost import XGBClassifier, XGBRegressor

        xg_grid = {"max_depth": [2, 3], "learning_rate": [0.05, 0.1], "n_estimators": [100]} \
            if not quick else {"max_depth": [3], "n_estimators": [50]}
        L["XGBoost"] = {
            "reg": _tuned(XGBRegressor(random_state=SEED, n_jobs=1, verbosity=0), xg_grid),
            "clf": _tuned(XGBClassifier(random_state=SEED, n_jobs=1, verbosity=0,
                                        eval_metric="logloss"), xg_grid),
        }
    except ImportError:
        log.warning("xgboost not installed -> XGBoost learner skipped (pip install xgboost)")
    return L

# ===========================================
# 5. POSITIVITY / OVERLAP  (Figure S1)
# ================================================
def overlap_check(df: pd.DataFrame, out: Path) -> None:
    from sklearn.ensemble import GradientBoostingClassifier

    X, d = df[BASELINE_COVARIATES], df["D_mlcc"]
    ps = cross_val_predict(
        GradientBoostingClassifier(random_state=SEED, max_depth=2, n_estimators=100),
        X, d, cv=N_FOLDS, method="predict_proba")[:, 1]

    summary = {
        "ps_min": ps.min(), "ps_max": ps.max(),
        "n_below_0.05": int((ps < 0.05).sum()), "n_above_0.95": int((ps > 0.95).sum()),
        "ps_min_treated": ps[d == 1].min(), "ps_max_control": ps[d == 0].max(),
    }
    pd.Series(summary).to_csv(out / "figS1_overlap_summary.csv", header=["value"])
    log.info("Propensity range [%.3f, %.3f]; <0.05: %d; >0.95: %d",
             ps.min(), ps.max(), summary["n_below_0.05"], summary["n_above_0.95"])

    fig, ax = plt.subplots(figsize=(6, 4))
    bins = np.linspace(0, 1, 31)
    ax.hist(ps[d == 1], bins, alpha=0.6, label="MLCC", color="tab:blue")
    ax.hist(ps[d == 0], bins, alpha=0.6, label="Routine care", color="tab:red")
    ax.set(xlabel="Estimated P(MLCC | X)", ylabel="Women", title="Propensity-score overlap")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "figS1_overlap.png", dpi=300)
    plt.close(fig)
# ======================================
# 6. DML ESTIMATION  (Table 2)
# ===========================================
def _summarise(model_name: str, learner: str, dml) -> dict:
    theta = float(np.ravel(dml.coef)[0])
    se = float(np.ravel(dml.se)[0])
    t = float(np.ravel(dml.t_stat)[0])
    p = float(np.ravel(dml.pval)[0])
    ci = np.asarray(dml.confint(level=0.95)).ravel()
    return {
        "Model": model_name, "Learner": learner,
        "Coef (reduction in P(preterm))": -theta,
        "Std err": se, "t": -t, "P-value": p,
        "2.5%": -ci[1], "97.5%": -ci[0],
        "Raw ATE (Y=1 preterm)": theta,
    }

def run_dml_ate(df: pd.DataFrame, learners: dict, n_reps: int) -> pd.DataFrame:
    import doubleml as dml  # imported here so earlier steps run without it
    np.random.seed(SEED)
    covs = list(BASELINE_COVARIATES)
    z_col = INSTRUMENT if (INSTRUMENT and INSTRUMENT in df.columns) else None
    if INSTRUMENT and z_col is None:
        log.warning("INSTRUMENT '%s' not found in data -> IV models skipped.", INSTRUMENT)
    if z_col is None:
        log.warning("No instrument configured -> PLIV / IIVM skipped (set INSTRUMENT).")
    if z_col is not None and z_col in covs:
        covs.remove(z_col)
    data = dml.DoubleMLData(df, y_col="Y_preterm", d_cols="D_mlcc", x_cols=covs)
    data_iv = (dml.DoubleMLData(df, y_col="Y_preterm", d_cols="D_mlcc", x_cols=covs, z_cols=z_col)
               if z_col else None)
    results = []
    for name, L in learners.items():
        reg, clf = L["reg"], L["clf"]
        specs = [
            ("PLR", lambda: dml.DoubleMLPLR(data, ml_l=clone(reg), ml_m=clone(reg),
                                             n_folds=N_FOLDS, n_rep=n_reps)),
            ("IRM", lambda: dml.DoubleMLIRM(data, ml_g=clone(reg), ml_m=clone(clf),
                                             n_folds=N_FOLDS, n_rep=n_reps, score="ATE",
                                            trimming_threshold=0.01)),
        ]
        if data_iv is not None:
            specs += [
                ("PLIV", lambda: dml.DoubleMLPLIV(data_iv, ml_l=clone(reg), ml_m=clone(reg),
                                                   ml_r=clone(reg), n_folds=N_FOLDS, n_rep=n_reps)),
                ("IIVM", lambda: dml.DoubleMLIIVM(data_iv, ml_g=clone(reg), ml_m=clone(clf),
                                                   ml_r=clone(clf), n_folds=N_FOLDS, n_rep=n_reps,
                                                   trimming_threshold=0.01)),
            ]
        for mname, builder in specs:
            try:
                model = builder().fit()
                results.append(_summarise(mname, name, model))
                log.info("%-5s | %-13s | coef=%.4f se=%.4f", mname, name,
                         results[-1]["Coef (reduction in P(preterm))"], results[-1]["Std err"])
            except Exception as exc:  # keep going if a single fit fails
                log.error("%s / %s failed: %s", mname, name, exc)
    return pd.DataFrame(results)

# ==========================================
# 7. QUANTILE TREATMENT EFFECTS  (Table 3)
# ==========================================
def run_qte(df: pd.DataFrame, learners: dict, n_reps: int,
            outcome: str = "Y_preterm", quantiles=(0.25, 0.50, 0.75))

   ## NOTE: for a binary outcome the 25/50/75th quantiles of the outcome are all 0
    import doubleml as dml
    if df[outcome].nunique() <= 2:
        log.warning("QTE outcome '%s' is binary -> quantile effects are degenerate/uninformative. "
                    "Use a continuous outcome for a meaningful QTE.", outcome)
    L = learners.get("Random forest") or next(iter(learners.values()))
    data = dml.DoubleMLData(df, y_col=outcome, d_cols="D_mlcc", x_cols=list(BASELINE_COVARIATES))
    rows = []
    try:
        qte = dml.DoubleMLQTE(data, ml_g=clone(L["clf"]), ml_m=clone(L["clf"]),
                              quantiles=list(quantiles), score="PQ",
                              n_folds=N_FOLDS, n_rep=n_reps).fit()
        ci = qte.confint(level=0.95)
        for i, q in enumerate(quantiles):
            est = float(qte.coef[i])
            rows.append({
                "Quantile": f"Q{i + 1} ({q:.2f})",
                "Estimate (reversed sign)": -est,
                "Std err": float(qte.se[i]),
                "t": -float(qte.t_stat[i]),
                "P-value": float(qte.pval[i]),
                "2.5%": -float(ci.iloc[i, 1]),
                "97.5%": -float(ci.iloc[i, 0]),
            })
    except Exception as exc:
        log.error("QTE failed: %s", exc)
    return pd.DataFrame(rows)

# =======================
# 8. MAIN
# =======================
def parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", default=os.environ.get("MLCC_DATA"),
                    help="Path to the Excel/CSV file (or set MLCC_DATA).")
    ap.add_argument("--out", default="results", help="Output directory for tables/figures.")
    ap.add_argument("--quick", action="store_true", help="Tiny grids and 1 repetition (smoke test).")
    ap.add_argument("--skip-dml", action="store_true", help="Run only Steps 1-4.")
    return ap.parse_args()


def main() -> None:
    args = parse_args()
    if not args.data:
        raise SystemExit("Provide --data PATH or set the MLCC_DATA environment variable.")
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    warnings.filterwarnings("ignore")
    np.random.seed(SEED)

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    n_reps = 1 if args.quick else N_REPS
    df = load_data(Path(args.data))
    table1(df).to_csv(out / "table1_descriptives.csv", index=False)
    predictive_model(df, out)
    overlap_check(df, out)
    if args.skip_dml:
        return
    learners = make_learners(quick=args.quick)
    t2 = run_dml_ate(df, learners, n_reps)
    t2.to_csv(out / "table2_dml_ate.csv", index=False)
    t3 = run_qte(df, learners, n_reps)
    t3.to_csv(out / "table3_qte.csv", index=False)

    with pd.option_context("display.width", 200, "display.max_columns", 20, "display.precision", 4):
        print("\n=== Table 2: DML ATE (positive = lower P(preterm) with MLCC) ===")
        print(t2.drop(columns=["Raw ATE (Y=1 preterm)"]).to_string(index=False))
        print("\n=== Table 3: Quantile treatment effects ===")
        print(t3.to_string(index=False))
    log.info("All outputs written to %s", out.resolve())
